In [4]:
#Step 1: Load & Structure Data#
import pandas as pd
import numpy as np

df = pd.read_csv("credit_risk_dataset.csv")

print(df.shape)
print(df.info())

(32581, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB
None


In [6]:
#🔹 Loan to Income Ratio#
df['loan_income_ratio'] = df['loan_amnt']/df['person_income']

In [7]:
#🔹 Interest Burden Score#
df['loan_interest_burden'] = df['loan_int_rate']/df['loan_percent_income']

In [8]:
#🔹 Employment Stability Flag#
df['stable_employment'] = np.where(df['person_emp_length'] >= 3,1,0)

In [11]:
# Step 3: Build BNPL Risk Score Model Weighted model (realistic approach):#

df['risk_score'] = (
    0.30 * df['loan_income_ratio'] +
    0.25 * df['loan_int_rate'] + 
    0.20 * df['loan_percent_income'] +
    0.15 * (1 - df['stable_employment']) +
    0.10 * (1 / df['person_age'])
)

In [12]:
#Step 4: Risk Bands (Production Style)#

df['risk_band'] = pd.qcut(
    df['risk_score'],
    q=5,
    labels=['Very Low','Low','Medium','High','Very High']
)
    

In [13]:
#Step 5: Expected Loss Calculation (FinTech Standard)
# Expected Loss = PD × LGD × EAD

df['PD'] = df['loan_status']
df['LGD'] = 0.6
df['EAD'] = df['loan_amnt']

df['Expected_loss'] = df['PD'] * df['LGD'] * df['EAD']



In [16]:
#Step 6: Approval Strategy Simulation

threshold = df['risk_score'].quantile(0.75)

df['credit_decision'] = np.where(
    df['risk_score'] > threshold,
    "Reject",
    "Approve"
)

#Now calculate impact:

approval_rate = (df['credit_decision'] == "Approve").mean()
default_rate = df[df['credit_decision'] == "Approve"]['loan_status'].mean()

In [19]:
# Step 7: Export Final Dataset to SQL 
df.to_csv("bnpl_risk_featured.csv", index = False)